# 12 — Does finance distinguish viable from non-viable candidates? (2024)

This notebook is a focused viability zoom.

**Question:** Do viable candidates generally have larger fundraising and
spending profiles than non-viable candidates, and does that relationship vary
by district?

### Important methodological caveat

In this project, viability is defined from **ballot mentions relative to the
district STV threshold**.

Therefore:

`finance → mentions`

and

`finance → viability`

are two views of related information, **not two independent confirmations**.


## 1. Setup

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.api as sm
from IPython.display import display

pd.set_option("display.max_columns", 100)

# Find the repository root from either the repo root or notebooks/.
cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    ROOT = cwd
elif (cwd.parent / "pyproject.toml").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not find pyproject.toml in this folder or its parent."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from helpers.paths import (
    PROCESSED,
    fundraising_processed_dir,
    spending_processed_dir,
)

YEAR = 2024
CONTEST = "city_council"

print("ROOT:", ROOT)


ROOT: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis


## 2. Load the candidate table from Notebook 10

In [2]:
topline_path = (
    PROCESSED
    / "finance_analysis"
    / str(YEAR)
    / "question_3"
    / "topline"
    / "candidate_topline_finance_support.csv"
)

analysis = pd.read_csv(
    topline_path
)

analysis["viable_int"] = (
    analysis["is_viable"]
    .astype(bool)
    .astype(int)
)

analysis["log_fundraising"] = np.log1p(
    analysis["fundraising"]
)

analysis["log_spending"] = np.log1p(
    analysis["total_spending"]
)

print("Candidates:", len(analysis))


Candidates: 98


## 3. How different are the finance distributions?

Medians are especially useful here because campaign-finance data have large
outliers.


In [3]:
viability_summary = (
    analysis
    .groupby(
        "is_viable",
        as_index=False,
    )
    .agg(
        candidates=("candidate_key", "size"),
        median_fundraising=("fundraising", "median"),
        mean_fundraising=("fundraising", "mean"),
        median_contribution_count=(
            "contribution_count",
            "median",
        ),
        median_spending=("total_spending", "median"),
        mean_spending=("total_spending", "mean"),
        median_expenditure_count=(
            "expenditure_count",
            "median",
        ),
    )
)

display(
    viability_summary.round(0)
)

nonviable_median = viability_summary.loc[
    viability_summary["is_viable"].eq(False),
    "median_fundraising",
].iloc[0]

viable_median = viability_summary.loc[
    viability_summary["is_viable"].eq(True),
    "median_fundraising",
].iloc[0]

print(
    "Viable / non-viable median fundraising ratio:",
    round(
        viable_median / nonviable_median,
        1,
    ),
)


,is_viable,candidates,median_fundraising,mean_fundraising,median_contribution_count,median_spending,mean_spending,median_expenditure_count
0,False,69,7281.0,13988.0,276.0,16523.0,36051.0,78.0
1,True,29,49453.0,53902.0,1055.0,136928.0,233219.0,192.0


Viable / non-viable median fundraising ratio: 6.8


### Finding from the current run

The current medians are roughly **$49,453 for viable candidates versus
$7,281 for non-viable candidates**, about a **6.8× gap**.

That is strong descriptive evidence that finance and competitiveness are
related, but it is not a causal estimate.


## 4. Interactive candidate view

In [4]:
plot_data = analysis.dropna(
    subset=["fundraising"]
).copy()

fig = px.box(
    plot_data,
    x="is_viable",
    y="fundraising",
    points="all",
    hover_name="canonical_candidate",
    hover_data=[
        "district",
        "mentions",
        "first_place_votes",
        "contribution_count",
    ],
    log_y=True,
    labels={
        "is_viable": "Viable",
        "fundraising": "Fundraising ($, log scale)",
    },
    title="Fundraising distribution: viable vs. non-viable",
)

fig.show()


## 5. Point-biserial correlations

Pearson correlation between a continuous variable and a 0/1 variable is also
called a **point-biserial correlation**.

We calculate it pooled and by district.


In [5]:
finance_measures = [
    "fundraising",
    "total_spending",
    "contribution_count",
    "expenditure_count",
]

rows = []

for finance in finance_measures:
    pair = analysis[
        ["viable_int", finance]
    ].dropna()

    if pair["viable_int"].nunique() == 2:
        rows.append(
            {
                "scope": "all districts",
                "district": np.nan,
                "finance_measure": finance,
                "n": len(pair),
                "correlation": pair[
                    "viable_int"
                ].corr(
                    pair[finance]
                ),
            }
        )

    for district in sorted(
        analysis["district"].dropna().unique()
    ):
        pair = analysis.loc[
            analysis["district"].eq(district),
            ["viable_int", finance],
        ].dropna()

        if (
            len(pair) >= 3
            and pair["viable_int"].nunique() == 2
        ):
            rows.append(
                {
                    "scope": "district",
                    "district": district,
                    "finance_measure": finance,
                    "n": len(pair),
                    "correlation": pair[
                        "viable_int"
                    ].corr(
                        pair[finance]
                    ),
                }
            )

viability_correlations = pd.DataFrame(
    rows
)

display(
    viability_correlations.round(3)
)


,scope,district,finance_measure,n,correlation
0,all districts,NaN,fundraising,65,0.705
1,district,1.0,fundraising,13,0.835
2,district,2.0,fundraising,20,0.669
3,district,3.0,fundraising,15,0.919
4,district,4.0,fundraising,17,0.506
5,all districts,NaN,total_spending,65,0.438
6,district,1.0,total_spending,11,0.433
7,district,2.0,total_spending,19,0.761
8,district,3.0,total_spending,16,0.688
9,district,4.0,total_spending,19,0.672


### Finding from the current run

Fundraising–viability correlation is about **0.705 pooled**, with clear
district heterogeneity. In the current data it is about **0.835 in D1,
0.919 in D3, and 0.506 in D4**.


## 6. Descriptive linear probability model

We fit:

`viable (0/1) = a + b × log(1 + money)`

This is easy to read but is **not a final classifier**. A linear probability
model can predict values below 0 or above 1.

Its value here is only to compare the strength of the relationship across
districts.


In [6]:
def viability_model(data, money, district=None):
    if district is None:
        pair = data[
            ["viable_int", money]
        ].dropna()

        scope = "all districts"

    else:
        pair = data.loc[
            data["district"].eq(district),
            ["viable_int", money],
        ].dropna()

        scope = "district"

    if (
        len(pair) < 4
        or pair["viable_int"].nunique() < 2
    ):
        return None

    x = np.log1p(
        pair[money].to_numpy(dtype=float)
    )

    y = pair[
        "viable_int"
    ].to_numpy(dtype=float)

    model = sm.OLS(
        y,
        sm.add_constant(x),
    ).fit()

    return {
        "scope": scope,
        "district": district,
        "money": money,
        "n": int(model.nobs),
        "p_value": model.pvalues[1],
        "r_squared": model.rsquared,
    }


model_rows = []

for money in [
    "fundraising",
    "total_spending",
]:
    pooled = viability_model(
        analysis,
        money,
    )

    if pooled is not None:
        model_rows.append(
            pooled
        )

    for district in sorted(
        analysis["district"].dropna().unique()
    ):
        result = viability_model(
            analysis,
            money,
            district=district,
        )

        if result is not None:
            model_rows.append(
                result
            )

viability_models = pd.DataFrame(
    model_rows
)

display(
    viability_models.round(
        {
            "p_value": 6,
            "r_squared": 3,
        }
    )
)


,scope,district,money,n,p_value,r_squared
0,all districts,NaN,fundraising,65,0.000000,0.454
1,district,1.0,fundraising,13,0.000014,0.833
2,district,2.0,fundraising,20,0.001916,0.423
3,district,3.0,fundraising,15,0.000101,0.700
4,district,4.0,fundraising,17,0.024572,0.294
5,all districts,NaN,total_spending,65,0.000000,0.454
6,district,1.0,total_spending,11,0.000204,0.800
7,district,2.0,total_spending,19,0.004582,0.385
8,district,3.0,total_spending,16,0.001198,0.539
9,district,4.0,total_spending,19,0.003490,0.403


In [7]:
district_models = viability_models[
    viability_models["scope"].eq("district")
].copy()

district_models["district"] = (
    district_models["district"]
    .astype(int)
    .astype(str)
)

fig = px.bar(
    district_models,
    x="district",
    y="r_squared",
    color="money",
    barmode="group",
    hover_data=["n", "p_value"],
    labels={
        "district": "District",
        "r_squared": "R²",
        "money": "Finance measure",
    },
    title="Finance–viability relationship varies by district",
)

fig.show()


## 7. Conclusion

**What this zoom adds:**

1. Viable candidates have much larger median finance profiles.
2. The relationship is not equally strong in every district.
3. D4 is notably weaker for fundraising than D1 and D3 in the current run.
4. This does **not** independently confirm Notebook 10, because viability is
   constructed from mentions.

This notebook should remain descriptive. Candidate-level explanations belong
in Notebook 13, where we explicitly identify cases for qualitative follow-up.


## 8. Export

In [8]:
output_dir = (
    PROCESSED
    / "finance_analysis"
    / str(YEAR)
    / "question_3"
    / "viability"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

viability_summary.to_csv(
    output_dir / "viability_group_summary.csv",
    index=False,
)

viability_correlations.to_csv(
    output_dir / "viability_correlations.csv",
    index=False,
)

viability_models.to_csv(
    output_dir / "viability_linear_probability_models.csv",
    index=False,
)

print("SAVED:", output_dir)


SAVED: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/finance_analysis/2024/question_3/viability
